In [1]:
import pandas as pd
import json

from typing import Tuple, List

In [2]:
# load sets to list from json file
class LoadJson:

    def __init__(self, file_name: str)-> None:
        self.file_name = file_name

    def _load(self) -> None:
        with open(file_name, 'r') as file:
            self.json_data = json.load(file)

    def _clean(self) -> None:
        index_ls = []
        set_ls = []
        for json_set in self.json_data:
            name = json_set.keys()
            set_info = json_set.values()

            index_ls.append(*name)
            set_ls.append(*set_info)

        self.index_ls = index_ls
        self.set_ls = set_ls

    def __call__(self) -> Tuple[List]:
        self._load()
        self._clean()

        return self.index_ls, self.set_ls

file_name = 'scraped_brickeconomy_final.json'
loader: LoadJson = LoadJson(file_name)
index_ls, set_ls = loader()


In [3]:
# transform neasted dictionaries to flatten dictionary 
class Transform:

    def __init__(self, index_ls, set_ls) -> None:
        self.index_ls = index_ls
        self.set_ls = set_ls

    def _flatten_dict_set_info(self, dictionary: dict) -> dict:
        temp_dict = {}

        theme = dictionary['set_info'].get('theme', None)
        year = dictionary['set_info'].get('year', None)
        av = dictionary['set_info'].get('availability', None)
        pieces = dictionary['set_info'].get('Pieces', None)
        minifigs = dictionary['set_info'].get('Minifigs', None)

        temp_dict['Theme'] = theme
        temp_dict['Year'] = year
        temp_dict['Availability'] = av
        temp_dict['Pieces'] = pieces
        temp_dict['Minifigs'] = minifigs

        return temp_dict

    def _flatten_dict_prices(self, dictionary: dict) -> dict:
        temp_dict = {}

        retail = dictionary['prices'].get('Retail', None)
        value = dictionary['prices'].get('Value', None)

        temp_dict['Retail'] = retail
        temp_dict['Value'] = value

        return temp_dict

    def _convert_to_dict(self, stores: list) -> dict:
        temp_dict = {'LEGO': 0, 'Amazon': 0, 'Bricklink': 0, 'StockX': 0, 'eBay': 0}

        for store in stores:
            match store:
                case 'LEGO':
                    temp_dict['LEGO'] = 1
                case 'StockX':
                    temp_dict['StockX'] = 1
                case 'Amazon':
                    temp_dict['Amazon'] = 1
                case 'Bricklink':
                    temp_dict['Bricklink'] = 1
                case _:
                    temp_dict['eBay'] = 1 
        return temp_dict
    
    def Process(self) -> None:

        list_to_df = []
        error_ls = [] # for potential missing informations -> finally only set_info
        for num, lego_set in enumerate(self.set_ls):
            set_dict = {}

            try:
                temp = self._flatten_dict_set_info(lego_set)
                set_dict.update(temp)
            except:
                # print(f'An error occurred for set_info - iteration number: {num}')
                error_ls.append(num)

            try:
                temp = self._flatten_dict_prices(lego_set)
                set_dict.update(temp)
            except:
                # print(f'An error occurred for prices - iteration number: {num}')
                error_ls.append(num)
            
            try:
                temp = self._convert_to_dict(lego_set['stores'])
                set_dict.update(temp)
            except:
                # print(f'An error occurred for stories - iteration number: {num}')
                error_ls.append(num)

            list_to_df.append(set_dict)

        self.list_to_df = list_to_df
        self.error_ls = error_ls
        self.DisplayErrors()

    def DisplayErrors(self) -> None:

        for index in self.error_ls:
            print(f'The error occured for set index: {index}\n{self.set_ls[index]}\n')

    def RemoveMissingElements(self) -> None:

        for index in sorted(self.error_ls, reverse=True):
            del self.list_to_df[index]
            del self.index_ls[index]

transform: Transform = Transform(index_ls, set_ls)
transform.Process()
transform.RemoveMissingElements()




The error occured for set index: 442
{'set_info': None, 'stores': ['eBay'], 'prices': {'Retail': 'Promotional or Unknown', 'Value': 167.2}}

The error occured for set index: 761
{'set_info': None, 'stores': ['Bricklink'], 'prices': {'Retail': 339.99, 'Value': 545.81}}



In [4]:
# convert list of dicts to dataframe

set_df = pd.DataFrame(transform.list_to_df, index=transform.index_ls)
set_df # check missing values using Data Wrangler

# missing pieces -> promotional minifig -> can be set to zero
# missinc minifigs -> should be set to zero

# Change Availability column to numeric
qry = (set_df['Availability'] == 'Exclusive')
set_df['Exclusive'] = 0
set_df.loc[qry, 'Exclusive'] = 1

# Fill missing pieces
set_df.loc[set_df['Pieces'].isna(), 'Pieces'] = 0

# Fill missing minifigs
set_df.loc[set_df['Minifigs'].isna(), 'Minifigs'] = 0

# Fill promotional sets with retail zero and create promotional column
qry = (set_df['Retail'].isin(['Promotional', 'Promotional or Unknown']))
set_df['Promotional'] = 0
set_df.loc[qry, 'Promotional'] = 1
set_df.loc[qry, 'Retail'] = 0.00

# Fill value of sets on market
qry = (set_df['Value'].isin(['Not yet released', 'Available at retail']))
set_df.loc[qry, 'Value'] = set_df.loc[qry, 'Retail']

# Transform Year Column
set_df['Years'] = 2025 - set_df['Year']

# remove  Value Packs - mostly missing data about minifigs and pieces - it should be included - sets are based on 2 or more different sets...
qry = (set_df['Theme'].str.strip() != 'Value Packs')
set_df = set_df[qry]

# Drop columns and change datatype
set_df.drop(columns=['Theme', 'Year', 'Availability'], inplace=True)
set_df['Minifigs'] = set_df['Minifigs'].astype(int)
set_df['Retail'] = set_df['Retail'].astype(float)
set_df['Value'] = set_df['Value'].astype(float)

In [5]:
# drop promotional and exclusive sets
print(set_df.shape)
print(set_df['Promotional'].value_counts())

qry = ((set_df['Promotional'] != 1) & (set_df['Exclusive'] != 1))

set_df = set_df[qry]
set_df.drop(columns=['Promotional', 'Exclusive'], inplace=True)

print(set_df.shape)

(752, 12)
Promotional
0    658
1     94
Name: count, dtype: int64
(655, 10)


In [6]:
# Check head
set_df.head(n= 15)

,Pieces,Minifigs,Retail,Value,LEGO,Amazon,Bricklink,StockX,eBay,Years
75268 Snowspeeder,91,2,19.99,27.63,0,1,1,0,1,5
75235 X-wing Starfighter Trench Run,132,4,29.99,40.00,0,1,1,1,1,6
75237 TIE Fighter Attack,77,2,19.99,43.41,0,1,1,1,1,6
75247 Rebel A-wing Starfighter,62,2,14.99,29.23,0,1,1,0,1,6
75416 Chopper (C1-10P) Astromech Droid,1039,1,99.99,99.99,1,1,0,1,1,0
75412 Death Trooper & Night Trooper Battle Pack,119,4,22.99,22.99,1,0,0,0,1,0
75385 Battle on Peridea,382,5,54.99,48.17,0,1,0,1,1,1
75357 Ghost & Phantom II,1394,4,159.99,159.99,1,1,0,1,1,2
75364 New Republic E-wing vs. Shin Hati's Starfighter,1056,5,109.99,97.88,0,1,0,1,1,2
75362 Ahsoka Tano's T-6 Jedi Shuttle,599,4,79.99,65.00,0,1,0,1,1,2


In [7]:
# Check dtypes
set_df.dtypes

Pieces         int64
Minifigs       int64
Retail       float64
Value        float64
LEGO           int64
Amazon         int64
Bricklink      int64
StockX         int64
eBay           int64
Years          int64
dtype: object

In [8]:
# write to parquet
file_out_name = 'clean_BE_df.parquet'

set_df.to_parquet(file_out_name)

# write to excel
file_out_name = 'clean_BE_df.xlsx'

set_df.to_excel(file_out_name)